# Ungraded Lab: PyTorch

Welcome to the first ungraded lab of this module!

This lab is a short introduction to the PyTorch library. You will learn how PyTorch stores data using tensors, how tensors differ from Python lists, and how to create and inspect them. There is nothing to submit here. Just read through the explanations and run the code cells to build your intuition.

## Objectives

By the end of this lab, you will be able to:

- **Understand why PyTorch exists**: See how tensor operations outperform plain Python lists by orders of magnitude.
- **Initialize tensors**: Create tensors using `torch.tensor`, `torch.zeros`, `torch.arange`, `torch.randn`, and other constructors.
- **Inspect tensor dtypes**: Understand how PyTorch represents numbers with fixed bit-widths and why that matters.
- **Work with tensor shapes**: Read and interpret the `.shape` property for tensors of any dimensionality.
- **Index into tensors**: Use single indices, multi-dimensional indexing, and slicing to access tensor elements.

## Table of Contents

- Setup
- PyTorch Preliminary
- Tensor Initialization
- Tensor dtype
- Tensor shape
- Tensor indexing

## Setup

In [1]:
import torch

## PyTorch Preliminary

PyTorch provides data structures that perform math very quickly. It also allows us to put our data structures on the GPU (though we will not need GPU in this module). Below is a figure that compares matrix multiplies performed with Pure Python (nested lists) vs PyTorch data structure on CPU vs PyTorch data structure on GPU. Notice the factor of improvement between the different plots.

![Benchmark showing difference between python list vs PyTorch tensor on CPU vs PyTorch tensor on GPU. GPU tensors are significantly faster for large data sizes.](./benchmark.png)

> Matrix size: 256x256
> 
> Python lists (CPU):  mean = 764.1693 ms
> 
> PyTorch tensors (CPU): mean = 7.4989 ms
> 
> PyTorch tensors (GPU): mean = 0.0400 ms
> 
> Speedup between list vs PyTorch tensor on GPU: 19102.90x

## Tensors initialization

Why doesn't Python `list` perform fast natively? Why is there a need for a separate framework like PyTorch to store elements?

Python `list` prioritizes *flexibility* over speed. With the way that `list` is laid out in memory, it can easily be appended to, store multiple data types, and have a jagged structure.

```python
mylist = []
mylist.append(5)  # appending is allowed

mylist = [2.5, 'test string', [], {'test-dictionary': -1}]  # multiple data types are allowed

mylist = [  # jagged structure is allowed
    [1,2,3],
    [4,5],
    [6,7,8,9]
]
```

PyTorch on the other hand stores elements using "tensors". Most AI use-cases (and this tutorial) utilize something called "strided tensors" (a.k.a. "dense tensors") which are instances of `torch.Tensor`. We will not cover the alternative formats that PyTorch may support, and everything said about tensors here may not be applicable to those other formats. We will proceed by referring to "strided tensors" merely as "tensors" from now on.

There are several ways to initialize a tensor, but they require the size to be known ahead of time. All elements are typically literally next to each other in memory.
```python
mytensor = torch.tensor([1,2,3])  # initialize by passing in a list
mytensor = torch.zeros(5)         # initialize with five zeros
mytensor = torch.arange(5)        # initialize with 0,1,2,3,4
```

We have much less flexibility on how to store elements than we had with `list`.
```python
mytensor = torch.tensor([])
mytensor.append(5)  # AttributeError

mytensor = torch.tensor([2.5, 'test string', [], {'test-dictionary': -1}])  # TypeError

torch.tensor([  # ValueError
    [1,2,3],
    [4,5],
    [6,7,8,9]
])
```

However, because of the way elements are laid out in memory, peforming calculations on the elements of `torch.Tensor` can be done much faster. When necessary, you can also more easily operate on separate chunks of a `torch.Tensor` independently in parallel which is the kind of computation that GPUs specialize in.

> **A note on interval notation:** Some of PyTorch's random functions use mathematical interval notation like `[0, 1)`. A square bracket `[` means the endpoint is **included**, while a round parenthesis `)` means the endpoint is **excluded**. So `[0, 1)` means "any value from 0 up to but not including 1", and `[0, 100)` means "any integer from 0 up to but not including 100" (i.e. 0 through 99).

Run the following cell to see how `torch.Tensor` can be initialized.

In [2]:
def example_initialize_tensors():
    print(torch.tensor([1,2,3]))
    print(torch.zeros(5))
    print(torch.arange(5))
    print(torch.randn(5))                    # from standard normal distribution
    print(torch.rand(5))                     # from continuous uniform distribution [0, 1)
                                             #   [0, 1) means: includes 0, excludes 1
                                             #   "[" = inclusive, ")" = exclusive
    print(torch.randint(0, 100, (5,)))       # from discrete uniform distribution on integers [0, 100)
                                             #   [0, 100) means: includes 0, excludes 100
                                             #   so the possible values are 0, 1, 2, ..., 99
    print(torch.tensor([[1,2,3], [4,5,6]]))

example_initialize_tensors()

tensor([1, 2, 3])
tensor([0., 0., 0., 0., 0.])
tensor([0, 1, 2, 3, 4])
tensor([ 0.8943, -0.7333,  1.3269,  0.2404,  1.6953])
tensor([0.8011, 0.7831, 0.9820, 0.2782, 0.0851])
tensor([41, 69, 35, 99, 88])
tensor([[1, 2, 3],
        [4, 5, 6]])


## Tensor dtype

Because `torch.Tensor` stores all elements as the same type, you can call `.dtype` on the whole tensor to see the type of its contents instead of needing to write code defensively by checking each individual element. Also, the numeric types that PyTorch uses have fixed bit-widths which limits the possible values they can represent. This is in contrast to native Python `int` which can be arbitrarily large but which performs arithmetic much slower. Python's `float` does behave similarly to PyTorch's `torch.float64`, but 64 bits is usually too much unnecessary space for AI models. PyTorch defaults to 32 bits for floating point numbers, and it is common practice to use even less precision in industry grade models.

In [3]:
def example_dtype():
    a = torch.tensor([1,2,3])
    print(f'{a = }\n{a.dtype = }\n')  # defaults to integer with 64 bits

    a = torch.tensor([1.0,2.0,3.0])
    print(f'{a = }\n{a.dtype = }\n')  # defaults to floating point with 32 bits

    # This is the maximum value for an 8-bit integer
    a = 2**7 - 1
    print(f'Native Python `int`: {a = }')
    a = torch.tensor(a, dtype=torch.int8)  # single value tensor
    print(f'PyTorch `int8`: {a = }\n')

    a = 2**7
    print(f'Native Python `int`: {a = }')
    try:
        # Causes an overflow, so we catch the error
        a = torch.tensor(a, dtype=torch.int8)
    except Exception as e:
        print(f'PyTorch `int8`: ...RuntimeError: {e}\n')

    # This is the maximum value for an 64-bit integer
    a = 2**63 - 1
    print(f'Native Python `int`: {a = }')
    a = torch.tensor(a, dtype=torch.int64)
    print(f'PyTorch `int64`: {a = }\n')

    a = 2**63
    print(f'Native Python `int`: {a = }')
    try:
        # Causes an overflow, so we catch the error
        a = torch.tensor(a, dtype=torch.int64)
    except Exception as e:
        print(f'PyTorch `int64`: ...RuntimeError: {e}')

example_dtype()

a = tensor([1, 2, 3])
a.dtype = torch.int64

a = tensor([1., 2., 3.])
a.dtype = torch.float32

Native Python `int`: a = 127
PyTorch `int8`: a = tensor(127, dtype=torch.int8)

Native Python `int`: a = 128
PyTorch `int8`: ...RuntimeError: value cannot be converted to type int8 without overflow

Native Python `int`: a = 9223372036854775807
PyTorch `int64`: a = tensor(9223372036854775807)

Native Python `int`: a = 9223372036854775808
PyTorch `int64`: ...RuntimeError: Overflow when unpacking long long


## Tensor shape

A `torch.Tensor` has a `.shape` property which contains the length of each axis.

In [4]:
def example_shape():
    a = torch.tensor(17)
    print(f'{a = }\n{a.shape = }\n')

    a = torch.tensor([17])
    print(f'{a = }\n{a.shape = }\n')

    a = torch.tensor([17, 18, 19])
    print(f'{a = }\n{a.shape = }\n')

    a = torch.tensor([[17, 18, 19], [20, 21, 22]])
    print(f'{a = }\n{a.shape = }\n')

    a = torch.tensor([[[17, 18, 19], [20, 21, 22]], [[23, 24, 25], [26, 27, 28]], [[29, 30, 31], [32, 33, 34]], [[35, 36, 37], [38, 39, 40]]])
    print(f'{a = }\n{a.shape = }')

example_shape()

a = tensor(17)
a.shape = torch.Size([])

a = tensor([17])
a.shape = torch.Size([1])

a = tensor([17, 18, 19])
a.shape = torch.Size([3])

a = tensor([[17, 18, 19],
        [20, 21, 22]])
a.shape = torch.Size([2, 3])

a = tensor([[[17, 18, 19],
         [20, 21, 22]],

        [[23, 24, 25],
         [26, 27, 28]],

        [[29, 30, 31],
         [32, 33, 34]],

        [[35, 36, 37],
         [38, 39, 40]]])
a.shape = torch.Size([4, 2, 3])


## Tensor indexing

You can index into a `torch.Tensor` similar to a `list`. If there are `N` axes, that means that `N` indices must be provided to specify a single element.

In [5]:
def example_indexing():
    a = torch.tensor([[1,2,3], [4,5,6], [7,8,9], [10,11,12]])
    print(f'{a = }')
    print()

    # The shape has two axes (i.e. len(a.shape) == 2), so we need two indices to specify a single element
    print(f'{a[0][0] = }')
    print(f'{a[0][1] = }')
    print(f'{a[1][0] = }')
    print(f'{a[-1][-1] = }')
    print()

    # It is preferrable to index using a single pair of brackets with indices separated by commas
    print(f'{a[0, 0] = }')
    print(f'{a[1, 0] = }')
    print(f'{a[0, 1] = }')
    print(f'{a[-1, -1] = }')
    print()

    # We can use slice syntax similar to Python lists
    print(f'{a[0:2, 0:2] = }')
    print(f'{a[0, :] = }')
    print(f'{a[:, 0] = }')
    print(f'{a[:, :] = }')

example_indexing()

a = tensor([[ 1,  2,  3],
        [ 4,  5,  6],
        [ 7,  8,  9],
        [10, 11, 12]])

a[0][0] = tensor(1)
a[0][1] = tensor(2)
a[1][0] = tensor(4)
a[-1][-1] = tensor(12)

a[0, 0] = tensor(1)
a[1, 0] = tensor(4)
a[0, 1] = tensor(2)
a[-1, -1] = tensor(12)

a[0:2, 0:2] = tensor([[1, 2],
        [4, 5]])
a[0, :] = tensor([1, 2, 3])
a[:, 0] = tensor([ 1,  4,  7, 10])
a[:, :] = tensor([[ 1,  2,  3],
        [ 4,  5,  6],
        [ 7,  8,  9],
        [10, 11, 12]])
